# RNN — ch4_household_power

Phase28: VALIDATION ONLY. Ba checkpoint train độc lập, serving NumPy CPU; không phải ba native GPU runtime. Notebook không train hoặc mở TEST. Thứ tự feature/lookback/horizons theo contracts Phase24–28; không dùng để khuyến nghị đầu tư.

In [ ]:
from pathlib import Path
import sys
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'TIEU_LUAN_AI_V3_EXECUTION_PLAN.md').exists())
if str(root) not in sys.path: sys.path.insert(0, str(root))
from scripts.demo_phase28 import sample_request, infer
from src.common.contracts import read_json
from src.ch04_rnn_lstm.protocol import BASE


In [ ]:
dataset = 'ch4_household_power'
request, sample_id = sample_request(dataset)
print('TRAIN sample:', sample_id)
print('Sequence shape:', len(request['input']['values']), len(request['input']['values'][0]))

In [ ]:
job = infer(request)
for result in job['results']:
    print(result['implementation'], result['actual_backend'], result['actual_device'], result['prediction'])
assert job['comparison_valid']

In [ ]:
model_id = job['results'][0]['model_id']
contract = read_json(BASE / 'runs' / model_id / 'model_contract.json')
print('Learnable layers:', sum(layer['kind'] in {'rnn','dense'} for layer in contract['layers']))
print(contract['composition_plain'])
print('Parameters:', contract['parameter_count'])

In [ ]:
release = read_json(BASE / 'release.json')
rows = [read_json(root / 'artifacts/metrics/phase28/validation' / (e['run_id'] + '.json')) for e in release['runs'] if e['dataset_id'] == dataset and e['fold'] is None]
print('Saved main VALIDATION records:', len(rows))
print('TEST was not executed. RNN/LSTM TEST belongs to Phase29.')
assert len(rows) == 11
assert not any(name in sys.modules for name in ('torch','tensorflow','keras'))